# DSFB Semiconductor End-to-End Notebook

This notebook runs the `dsfb-semiconductor` crate as a non-intrusive, read-only DSFB interpretation layer. It fetches or validates SECOM, probes PHM 2018, runs the SECOM benchmark, runs the PHM 2018 benchmark, renders the operator-facing figures, and surfaces the generated PDF and ZIP artifacts.

In [ ]:
from pathlib import Path
import json
import subprocess

CRATE = Path('/home/one/dsfb/crates/dsfb-semiconductor')
MANIFEST = CRATE / 'Cargo.toml'
DATA_ROOT = CRATE / 'data'
RAW_ROOT = DATA_ROOT / 'raw'
OUTPUT_ROOT = CRATE / 'output-dsfb-semiconductor'

def run_cmd(args, cwd=CRATE):
    print(' '.join(map(str, args)))
    completed = subprocess.run(args, cwd=cwd, check=True, text=True, capture_output=True)
    print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    return completed


In [ ]:
run_cmd(['cargo', 'run', '--manifest-path', str(MANIFEST), '--', 'fetch-secom', '--data-root', str(RAW_ROOT)])
run_cmd(['cargo', 'run', '--manifest-path', str(MANIFEST), '--', 'probe-phm2018', '--data-root', str(DATA_ROOT)])
run_cmd(['cargo', 'run', '--manifest-path', str(MANIFEST), '--', 'run-secom', '--data-root', str(RAW_ROOT), '--output-root', str(OUTPUT_ROOT)])

In [ ]:
secom_runs = sorted(OUTPUT_ROOT.glob('*_dsfb-semiconductor_secom'))
latest_secom = secom_runs[-1]
print(latest_secom)
run_cmd(['cargo', 'run', '--manifest-path', str(MANIFEST), '--', 'run-phm2018', '--data-root', str(DATA_ROOT), '--output-root', str(OUTPUT_ROOT), '--secom-run-dir', str(latest_secom)])
phm_runs = sorted(OUTPUT_ROOT.glob('*_dsfb-semiconductor_phm2018'))
latest_phm = phm_runs[-1]
print(latest_phm)
run_cmd(['cargo', 'run', '--manifest-path', str(MANIFEST), '--', 'render-non-intrusive-artifacts', '--run-dir', str(latest_secom)])
run_cmd(['cargo', 'run', '--manifest-path', str(MANIFEST), '--', 'render-unified-value-figure', '--secom-run-dir', str(latest_secom), '--phm-run-dir', str(latest_phm), '--paper-tex', str(CRATE / 'paper' / 'semiconductor.tex')])

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(latest_secom / 'figures' / 'dsfb_non_intrusive_architecture.png')))
display(Image(filename=str(latest_secom / 'figures' / 'dsfb_unified_value_figure.png')))

artifacts = {
    'report_pdf': str(latest_secom / 'report.pdf'),
    'results_zip': str(latest_secom / 'results.zip'),
    'episode_summary_csv': str(latest_secom / 'dsfb_episode_summary.csv'),
    'episode_precision_csv': str(latest_secom / 'dsfb_episode_precision.csv'),
    'recall_metrics_csv': str(latest_secom / 'dsfb_recall_metrics.csv')
}
print(json.dumps(artifacts, indent=2))